# Attention U-Net → BUSI — project notebook

**Crash-safe & resumable.** All outputs go to Google Drive, checkpoints are saved
every epoch, and finished work is skipped on re-run. If Colab disconnects, just
**re-run every cell top-to-bottom** — completed seeds are skipped and the
in-progress one resumes from its last epoch (all read back from Drive).

All logic lives in the `busi/` package; this notebook only orchestrates.
Control panel = `busi/config.py`.

## 1. Bootstrap — Colab only

On a fresh Colab runtime, run this to clone the repo + install deps. On a local
kernel (your own `.venv`) **skip this** — just run from the repo root.

In [1]:
import os, subprocess
repo = "/content/DL_attention_busi" if os.path.isdir("/content/DL_attention_busi") else "."
subprocess.run(["git", "-C", repo, "fetch", "origin"])
subprocess.run(["git", "-C", repo, "reset", "--hard", "origin/busi-extension"])
print(subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)
!grep -n ckpt_every {repo}/busi/config.py


grep: ./busi/config.py: No such file or directory


In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yuval-Naim/DL_attention_busi.git"   # public — no token
REPO_DIR = "DL_attention_busi"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt", "-i", "https://pypi.org/simple"], check=True)

print("cwd:", os.getcwd())

cwd: /content/DL_attention_busi


## 2. Mount Drive + set paths (the crash-safety step)

Everything that must survive a disconnect — dataset, checkpoints, results — lives
on **Drive**, never on the temporary Colab disk. Put your BUSI folders
(`benign/ malignant/ normal/`) under `DATA_ROOT`.

In [3]:
import os
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
else:
    DRIVE = "."                      # local run: keep outputs in the repo

DATA_ROOT = f"{DRIVE}/BUSI"          # your dataset (benign/ malignant/ normal/)
OUT_DIR   = f"{DRIVE}/busi_out"      # checkpoints + results persist here
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)
print("DATA_ROOT:", DATA_ROOT, "| exists:", os.path.isdir(DATA_ROOT))
print("OUT_DIR:  ", OUT_DIR)

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/BUSI | exists: True
OUT_DIR:   /content/drive/MyDrive/busi_out


## 3. Sanity + data check

Expected: **780** images (437/210/133). (If the data isn't on Drive yet, download
it from Kaggle — `aryashah2k/breast-ultrasound-images-dataset` — into `DATA_ROOT`.
Dataset: Al-Dhabyani et al., Data in Brief 2020, CC BY 4.0.)

In [4]:
import torch, busi
from busi.config import Config
from busi import train as T, experiment as E
from busi.data import list_busi_samples
print("busi", busi.__version__, "| torch", torch.__version__, "| device", T.get_device())

samples = list_busi_samples(DATA_ROOT, classes=("benign", "malignant", "normal"))
print("total samples:", len(samples))          # expect 780
split = E.get_or_make_split(Config(data_root=DATA_ROOT))
print({k: len(v) for k, v in split.items()})

busi 0.1.0 | torch 2.11.0+cu128 | device cuda
total samples: 780
{'train': 546, 'val': 117, 'test': 117}


## 4. Configure the run

Everything you'd tweak is in the **KNOBS** block at the top of the next cell —
`EPOCHS` (max; early-stopping usually ends sooner), `PATIENCE`, `SEEDS`, `MODELS`,
and `QUICK` (fast smoke). Defaults are PoC-appropriate: 50 epochs, patience 8,
3 seeds. Outputs point at Drive.

In [5]:
# ===== KNOBS — the few things you'll actually change =====
QUICK    = False               # True = fast smoke (1 seed, 3 epochs) to test the plumbing
EPOCHS   = 50                  # max epochs per run (early stopping usually ends sooner)
PATIENCE = 8                   # stop after this many epochs with no val-Dice improvement
SEEDS    = [42, 1]          # 3 -> report mean +- std; use [42, 1] to save Colab budget
MODELS   = ["unet", "attention_unet", "cbam_unet", "scse_unet"]  # drop some to save budget
# =========================================================

base = Config(
    data_root=DATA_ROOT,
    checkpoints_dir=f"{OUT_DIR}/checkpoints",   # -> Drive (best + resume ckpts)
    results_dir=f"{OUT_DIR}/results",           # -> Drive (per-seed + per-model JSON)
    epochs=(3 if QUICK else EPOCHS),
    early_stop_patience=PATIENCE,
    seeds=([42] if QUICK else SEEDS),
    ckpt_every=1,                               # save a resume checkpoint every epoch
)
print("epochs:", base.epochs, "| patience:", base.early_stop_patience,
      "| seeds:", base.seeds, "| out:", base.results_dir)

epochs: 50 | patience: 8 | seeds: [42, 1] | out: /content/drive/MyDrive/busi_out/results


## 5. Run — one model per cell (resumable)

Each cell trains all seeds for one model. If it crashes, **just re-run that cell**
(or the whole notebook) — finished seeds are skipped, the in-progress seed resumes
from its last epoch. Live per-epoch logs show progress.

In [6]:
E.run_seeds("unet", cfg=base)               # baseline


===== unet: seed 42 (1/2) =====
[unet_dice_ce_seed42] START
[fit] unet_dice_ce_seed42: up to 50 epochs on cuda
[unet_dice_ce_seed42] ep 1/50  loss=0.8075  val_dice=0.0000  spec=1.000  best=0.0000  1506s  *saved
[unet_dice_ce_seed42] ep 2/50  loss=0.7014  val_dice=0.0000  spec=1.000  best=0.0000  18s
[unet_dice_ce_seed42] ep 3/50  loss=0.6327  val_dice=0.4026  spec=0.900  best=0.4026  19s  *saved
[unet_dice_ce_seed42] ep 4/50  loss=0.5762  val_dice=0.5353  spec=0.700  best=0.5353  19s  *saved
[unet_dice_ce_seed42] ep 5/50  loss=0.5248  val_dice=0.4916  spec=1.000  best=0.5353  18s
[unet_dice_ce_seed42] ep 6/50  loss=0.5179  val_dice=0.5942  spec=0.750  best=0.5942  18s  *saved
[unet_dice_ce_seed42] ep 7/50  loss=0.5284  val_dice=0.5977  spec=0.700  best=0.5977  18s  *saved
[unet_dice_ce_seed42] ep 8/50  loss=0.4887  val_dice=0.5809  spec=0.550  best=0.5977  18s
[unet_dice_ce_seed42] ep 9/50  loss=0.4740  val_dice=0.5955  spec=0.350  best=0.5977  18s
[unet_dice_ce_seed42] ep 10/50  loss

{'model': 'unet',
 'seeds': [42, 1],
 'n_params': 2465058,
 'aggregate': {'lesion_dice': {'mean': 0.7281191242735097,
   'std': 0.0004995040976704113,
   'values': [0.7276196201758393, 0.7286186283711801]},
  'lesion_iou': {'mean': 0.635581939821263,
   'std': 0.0008146280190257094,
   'values': [0.6363965678402888, 0.6347673118022373]},
  'dice_benign': {'mean': 0.7442093856262396,
   'std': 0.006188229646008181,
   'values': [0.7503976152722478, 0.7380211559802314]},
  'dice_malignant': {'mean': 0.6938624388128589,
   'std': 0.014737904971309057,
   'values': [0.6791245338415499, 0.708600343784168]},
  'specificity': {'mean': 0.55, 'std': 0.1, 'values': [0.45, 0.65]},
  'fp_rate': {'mean': 0.45,
   'std': 0.10000000000000003,
   'values': [0.55, 0.35]}},
 'runs': [{'config': {'model_name': 'unet',
    'loss_name': 'dice_ce',
    'seed': 42,
    'data_root': '/content/drive/MyDrive/BUSI',
    'classes': ('benign', 'malignant', 'normal'),
    'img_size': 256,
    'split_ratios': (0.7, 

In [7]:
E.run_seeds("attention_unet", cfg=base)      # paper's additive gate


===== attention_unet: seed 42 (1/2) =====
[attention_unet_dice_ce_seed42] START
[fit] attention_unet_dice_ce_seed42: up to 50 epochs on cuda
[attention_unet_dice_ce_seed42] ep 1/50  loss=0.7329  val_dice=0.4002  spec=0.600  best=0.4002  19s  *saved
[attention_unet_dice_ce_seed42] ep 2/50  loss=0.5774  val_dice=0.5144  spec=0.800  best=0.5144  18s  *saved
[attention_unet_dice_ce_seed42] ep 3/50  loss=0.5283  val_dice=0.5281  spec=0.250  best=0.5281  19s  *saved
[attention_unet_dice_ce_seed42] ep 4/50  loss=0.5101  val_dice=0.5743  spec=0.400  best=0.5743  20s  *saved
[attention_unet_dice_ce_seed42] ep 5/50  loss=0.4900  val_dice=0.5865  spec=0.700  best=0.5865  20s  *saved
[attention_unet_dice_ce_seed42] ep 6/50  loss=0.4632  val_dice=0.6152  spec=0.500  best=0.6152  20s  *saved
[attention_unet_dice_ce_seed42] ep 7/50  loss=0.4394  val_dice=0.5768  spec=1.000  best=0.6152  20s
[attention_unet_dice_ce_seed42] ep 8/50  loss=0.4253  val_dice=0.5679  spec=0.950  best=0.6152  19s
[attention

{'model': 'attention_unet',
 'seeds': [42, 1],
 'n_params': 2705661,
 'aggregate': {'lesion_dice': {'mean': 0.7455216809568492,
   'std': 0.012117541810173327,
   'values': [0.7334041391466759, 0.7576392227670226]},
  'lesion_iou': {'mean': 0.6553472103323352,
   'std': 0.013186073370776041,
   'values': [0.6421611369615592, 0.6685332837031113]},
  'dice_benign': {'mean': 0.7795424797171367,
   'std': 0.013867227601248477,
   'values': [0.7656752521158883, 0.7934097073183852]},
  'dice_malignant': {'mean': 0.673090302951076,
   'std': 0.008392404319497171,
   'values': [0.6646978986315788, 0.6814827072705731]},
  'specificity': {'mean': 0.75,
   'std': 0.050000000000000044,
   'values': [0.7, 0.8]},
  'fp_rate': {'mean': 0.25,
   'std': 0.050000000000000044,
   'values': [0.30000000000000004, 0.19999999999999996]}},
 'runs': [{'config': {'model_name': 'attention_unet',
    'loss_name': 'dice_ce',
    'seed': 42,
    'data_root': '/content/drive/MyDrive/BUSI',
    'classes': ('benign', 

In [8]:
E.run_seeds("cbam_unet", cfg=base)           # desired


===== cbam_unet: seed 42 (1/2) =====
[cbam_unet_dice_ce_seed42] START
[fit] cbam_unet_dice_ce_seed42: up to 50 epochs on cuda
[cbam_unet_dice_ce_seed42] ep 1/50  loss=0.7172  val_dice=0.3617  spec=0.900  best=0.3617  18s  *saved
[cbam_unet_dice_ce_seed42] ep 2/50  loss=0.5661  val_dice=0.5566  spec=0.750  best=0.5566  18s  *saved
[cbam_unet_dice_ce_seed42] ep 3/50  loss=0.5081  val_dice=0.5871  spec=0.450  best=0.5871  19s  *saved
[cbam_unet_dice_ce_seed42] ep 4/50  loss=0.4916  val_dice=0.5304  spec=0.100  best=0.5871  19s
[cbam_unet_dice_ce_seed42] ep 5/50  loss=0.4670  val_dice=0.6183  spec=0.750  best=0.6183  19s  *saved
[cbam_unet_dice_ce_seed42] ep 6/50  loss=0.4268  val_dice=0.6769  spec=0.850  best=0.6769  19s  *saved
[cbam_unet_dice_ce_seed42] ep 7/50  loss=0.4334  val_dice=0.6404  spec=0.850  best=0.6769  18s
[cbam_unet_dice_ce_seed42] ep 8/50  loss=0.4029  val_dice=0.7007  spec=0.650  best=0.7007  19s  *saved
[cbam_unet_dice_ce_seed42] ep 9/50  loss=0.3692  val_dice=0.6783 

{'model': 'cbam_unet',
 'seeds': [42, 1],
 'n_params': 2468750,
 'aggregate': {'lesion_dice': {'mean': 0.7926472990484323,
   'std': 0.004269009434092197,
   'values': [0.7883782896143401, 0.7969163084825245]},
  'lesion_iou': {'mean': 0.7068380587388599,
   'std': 0.002052861128853889,
   'values': [0.704785197610006, 0.7088909198677138]},
  'dice_benign': {'mean': 0.8257645440683088,
   'std': 0.006295560873882167,
   'values': [0.8194689831944266, 0.832060104942191]},
  'dice_malignant': {'mean': 0.7221396161028887,
   'std': 4.5583953847350145e-05,
   'values': [0.722185200056736, 0.7220940321490413]},
  'specificity': {'mean': 0.575,
   'std': 0.12499999999999997,
   'values': [0.45, 0.7]},
  'fp_rate': {'mean': 0.42500000000000004,
   'std': 0.125,
   'values': [0.55, 0.30000000000000004]}},
 'runs': [{'config': {'model_name': 'cbam_unet',
    'loss_name': 'dice_ce',
    'seed': 42,
    'data_root': '/content/drive/MyDrive/BUSI',
    'classes': ('benign', 'malignant', 'normal'),


In [9]:
E.run_seeds("scse_unet", cfg=base)           # stretch


===== scse_unet: seed 42 (1/2) =====
[scse_unet_dice_ce_seed42] START
[fit] scse_unet_dice_ce_seed42: up to 50 epochs on cuda
[scse_unet_dice_ce_seed42] ep 1/50  loss=0.7554  val_dice=0.3733  spec=0.450  best=0.3733  18s  *saved
[scse_unet_dice_ce_seed42] ep 2/50  loss=0.5821  val_dice=0.4225  spec=0.950  best=0.4225  18s  *saved
[scse_unet_dice_ce_seed42] ep 3/50  loss=0.5334  val_dice=0.5640  spec=0.300  best=0.5640  18s  *saved
[scse_unet_dice_ce_seed42] ep 4/50  loss=0.4942  val_dice=0.5519  spec=0.850  best=0.5640  18s
[scse_unet_dice_ce_seed42] ep 5/50  loss=0.4763  val_dice=0.6134  spec=0.350  best=0.6134  18s  *saved
[scse_unet_dice_ce_seed42] ep 6/50  loss=0.4673  val_dice=0.6155  spec=0.800  best=0.6155  18s  *saved
[scse_unet_dice_ce_seed42] ep 7/50  loss=0.4460  val_dice=0.6137  spec=0.800  best=0.6155  18s
[scse_unet_dice_ce_seed42] ep 8/50  loss=0.4346  val_dice=0.6589  spec=0.700  best=0.6589  18s  *saved
[scse_unet_dice_ce_seed42] ep 9/50  loss=0.4228  val_dice=0.6312 

{'model': 'scse_unet',
 'seeds': [42, 1],
 'n_params': 2471385,
 'aggregate': {'lesion_dice': {'mean': 0.7579195683890882,
   'std': 0.006520287932059943,
   'values': [0.7644398563211482, 0.7513992804570283]},
  'lesion_iou': {'mean': 0.6759274772858082,
   'std': 0.009059492515076262,
   'values': [0.6849869698008844, 0.6668679847707318]},
  'dice_benign': {'mean': 0.7818233845021001,
   'std': 0.007927344608787912,
   'values': [0.789750729110888, 0.7738960398933121]},
  'dice_malignant': {'mean': 0.707027572793644,
   'std': 0.0035246188783808474,
   'values': [0.7105521916720249, 0.7035029539152632]},
  'specificity': {'mean': 0.775,
   'std': 0.025000000000000022,
   'values': [0.8, 0.75]},
  'fp_rate': {'mean': 0.22499999999999998,
   'std': 0.025000000000000022,
   'values': [0.19999999999999996, 0.25]}},
 'runs': [{'config': {'model_name': 'scse_unet',
    'loss_name': 'dice_ce',
    'seed': 42,
    'data_root': '/content/drive/MyDrive/BUSI',
    'classes': ('benign', 'maligna

## 6. Results table & figures

Built from whatever finished (works even if some models are still pending).

In [10]:
results = E.collect_results(MODELS, base.results_dir)
print(E.make_results_table(results))

# Attention-map figures for the attention model (best-seed checkpoint), saved to Drive.
if any(r["model"] == "attention_unet" for r in results):
    cfg_att = E._cfg_for(base, "attention_unet", base.seeds[0])
    ckpt = f"{cfg_att.checkpoints_dir}/{cfg_att.experiment_name}.pt"
    figs = E.save_prediction_figures(cfg_att, ckpt, f"{OUT_DIR}/figures", n=6)
    print("saved figures:", figs)

| Model | Params | Dice | IoU | Dice(ben) | Dice(mal) | Specificity |
|---|---|---|---|---|---|---|
| unet | 2.47M | 0.728±0.000 | 0.636±0.001 | 0.744±0.006 | 0.694±0.015 | 0.550±0.100 |
| attention_unet | 2.71M | 0.746±0.012 | 0.655±0.013 | 0.780±0.014 | 0.673±0.008 | 0.750±0.050 |
| cbam_unet | 2.47M | 0.793±0.004 | 0.707±0.002 | 0.826±0.006 | 0.722±0.000 | 0.575±0.125 |
| scse_unet | 2.47M | 0.758±0.007 | 0.676±0.009 | 0.782±0.008 | 0.707±0.004 | 0.775±0.025 |
saved figures: ['/content/drive/MyDrive/busi_out/figures/attention_unet_0.png', '/content/drive/MyDrive/busi_out/figures/attention_unet_1.png', '/content/drive/MyDrive/busi_out/figures/attention_unet_2.png', '/content/drive/MyDrive/busi_out/figures/attention_unet_3.png', '/content/drive/MyDrive/busi_out/figures/attention_unet_4.png', '/content/drive/MyDrive/busi_out/figures/attention_unet_5.png']


## 7. Notes

- **Recover from a crash:** re-run all cells. Done seeds skip; the in-progress one
  resumes from Drive — nothing lost beyond the current epoch.
- **Stretch — loss study** (Focal-Tversky on the best variant):
  ```python
  d = {**base.to_dict(), "loss_name": "focal_tversky", "experiment_name": ""}
  E.run_seeds("attention_unet", cfg=Config(**d))
  ```
- Prefer a GPU runtime (Runtime → Change runtime type → **T4 GPU**).